# EBM Sector ETF Triple Barrier Pipeline

Pipeline cuantitativo que implementa:

1. **Datos** — OHLCV diario de ETFs sectoriales y SPY (yfinance) + macro FRED
2. **Features** — Retorno, momentum, volatilidad, fuerza relativa vs SPY, volumen
3. **Neutralización cross-sectional** — Se resta la media entre sectores en cada fecha
4. **Triple Barrier Labeling** — TP/SL = 1× vol rolling 20d de SPY; barrera vertical = 7 sesiones
5. **EBM Classifier** — Explainable Boosting Machine (InterpretML)
6. **Purged Walk-Forward Expanding Window** — Validación 3 meses, purging evento
7. **PBO (CSCV)** — Probabilidad de Backtest Overfitting (López de Prado)
8. **Cluster Permutation Importance** — Importancia agrupada por correlación

> **OOS 2022-2026: INTOCABLE** — el modelo nunca ve estos datos durante el entrenamiento.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for CI
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from itertools import combinations

from sklearn.metrics import average_precision_score
from scipy.cluster.hierarchy import linkage, fcluster, dendrogram
from scipy.spatial.distance import squareform

from interpret.glassbox import ExplainableBoostingClassifier
import yfinance as yf

try:
    import pandas_datareader.data as web
    HAS_DATAREADER = True
except ImportError:
    HAS_DATAREADER = False
    print('pandas_datareader not available — skipping FRED data')

# ── Configuration ──────────────────────────────────────────────────────────────
ETFS        = ['XLE', 'XLB', 'XLI', 'XLK', 'XLF', 'XLP', 'XLY', 'XLV', 'XLU']
BENCHMARK   = 'SPY'
ALL_TICKERS = ETFS + [BENCHMARK]

START_DATE = '2005-01-01'
OOS_START  = '2022-01-01'
END_DATE   = '2025-12-31'

TP_MULT    = 1.0
SL_MULT    = 1.0
VOL_WINDOW = 20
VERT_BARS  = 7

VAL_MONTHS      = 3
MIN_TRAIN_YEARS = 3

PBO_SUBPERIODS = 8
FAST_MODE      = True   # set False for full-quality run

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('Setup complete.')

## 1. Descarga de datos

In [ ]:
# ── OHLCV ─────────────────────────────────────────────────────────────────────
print('Downloading OHLCV data from Yahoo Finance...')
raw = yf.download(ALL_TICKERS, start=START_DATE, end=END_DATE,
                  auto_adjust=True, progress=False)

close  = raw['Close'].dropna(how='all')
volume = raw['Volume'].dropna(how='all')
high   = raw['High'].dropna(how='all')
low    = raw['Low'].dropna(how='all')

print(f'  Shape : {close.shape}')
print(f'  Range : {close.index[0].date()} → {close.index[-1].date()}')
print(f'  Missing:\n{close.isna().sum()}')

# ── FRED macro ─────────────────────────────────────────────────────────────────
FRED_SERIES = {
    'FEDFUNDS' : 'fed_funds_rate',
    'GS10'     : 'treasury_10y',
    'BAA10Y'   : 'baa_credit_spread',
    'CPIAUCSL' : 'cpi',
    'UNRATE'   : 'unemployment',
}

macro_frames = []
if HAS_DATAREADER:
    print('\nDownloading FRED macro data...')
    for sid, name in FRED_SERIES.items():
        try:
            s = web.DataReader(sid, 'fred', START_DATE, END_DATE)
            s.columns = [name]
            macro_frames.append(s)
            print(f'  ✓ {sid}')
        except Exception as e:
            print(f'  ✗ {sid}: {e}')

if macro_frames:
    macro_raw = pd.concat(macro_frames, axis=1)
    macro = macro_raw.resample('B').last().reindex(close.index, method='ffill')
    if 'fed_funds_rate' in macro.columns:
        macro['fed_funds_chg'] = macro['fed_funds_rate'].diff(21)
    if 'treasury_10y' in macro.columns:
        macro['yield_chg'] = macro['treasury_10y'].diff(21)
    if 'cpi' in macro.columns:
        macro['cpi_yoy'] = macro['cpi'].pct_change(252) * 100
else:
    macro = pd.DataFrame(index=close.index)
    print('No macro features available — continuing without them.')

print(f'\nMacro features: {list(macro.columns)}')

## 2. Feature Engineering

In [ ]:
# ── SPY rolling vol (barrier size) ────────────────────────────────────────────
spy_ret      = close[BENCHMARK].pct_change()
spy_vol_roll = spy_ret.rolling(VOL_WINDOW).std()

etf_close  = close[ETFS]
etf_volume = volume[ETFS]
etf_high   = high[ETFS]
etf_low    = low[ETFS]
etf_ret    = etf_close.pct_change()

feature_groups = {}

# Returns
for p in [1, 5, 10, 20, 60]:
    feature_groups[f'ret_{p}d'] = etf_close.pct_change(p)

# Momentum
for p in [21, 63, 126, 252]:
    feature_groups[f'mom_{p}d'] = etf_close.pct_change(p)

# Realized volatility
for w in [10, 20, 60]:
    feature_groups[f'vol_{w}d'] = etf_ret.rolling(w).std()

# Relative strength vs SPY
for p in [5, 20, 60]:
    feature_groups[f'rs_spy_{p}d'] = (
        etf_close.pct_change(p).sub(close[BENCHMARK].pct_change(p), axis=0)
    )

# Volume ratio
feature_groups['vol_ratio'] = etf_volume / etf_volume.rolling(20).mean()

# Normalised true range proxy
feature_groups['atr_norm'] = (etf_high - etf_low) / etf_close

print(f'Feature groups  : {len(feature_groups)}')
print(f'Per-ETF features: {len(feature_groups)} × {len(ETFS)} ETFs')

## 3. Neutralización Cross-Sectional

In [ ]:
def cross_section_neutralize(df: pd.DataFrame) -> pd.DataFrame:
    """Subtract cross-sectional mean across ETFs at each date."""
    return df.sub(df.mean(axis=1), axis=0)

neutralized = {
    f'{name}_neu': cross_section_neutralize(df)
    for name, df in feature_groups.items()
}

print(f'Neutralized feature groups: {len(neutralized)}')

## 4. Triple Barrier Labeling

- **TP** = `1 × SPY rolling 20d std`
- **SL** = `1 × SPY rolling 20d std`
- **Vertical barrier** = 7 sesiones
- Label **+1** (TP hit), **−1** (SL hit), **0** (vertical barrier)

In [ ]:
def triple_barrier_label(close_arr, barrier_arr, tp_mult, sl_mult, vert_bars, dates):
    """
    Triple barrier labeling (vectorized inner loop).

    Returns
    -------
    labels     : ndarray  — 1 (TP), -1 (SL), 0 (vertical)
    event_ends : ndarray of datetime64
    """
    n = len(close_arr)
    labels     = np.full(n, np.nan)
    event_ends = np.empty(n, dtype='datetime64[ns]')
    event_ends[:] = np.datetime64('NaT')

    for i in range(n - vert_bars):
        if np.isnan(barrier_arr[i]) or np.isnan(close_arr[i]):
            continue

        p0 = close_arr[i]
        tp = tp_mult * barrier_arr[i]
        sl = sl_mult * barrier_arr[i]

        fut_r = (close_arr[i + 1: i + 1 + vert_bars] - p0) / p0
        valid = ~np.isnan(fut_r)

        tp_idx = np.where(valid & (fut_r >=  tp))[0]
        sl_idx = np.where(valid & (fut_r <= -sl))[0]

        first_tp = tp_idx[0] if len(tp_idx) else vert_bars
        first_sl = sl_idx[0] if len(sl_idx) else vert_bars

        if first_tp < first_sl:
            labels[i]     = 1
            event_ends[i] = np.datetime64(dates[min(i + 1 + first_tp, n - 1)])
        elif first_sl < first_tp:
            labels[i]     = -1
            event_ends[i] = np.datetime64(dates[min(i + 1 + first_sl, n - 1)])
        else:
            labels[i]     = 0
            event_ends[i] = np.datetime64(dates[min(i + vert_bars, n - 1)])

    return labels, event_ends


print('Computing triple barrier labels...')
spy_vol_arr = spy_vol_roll.values
dates_arr   = close.index

all_labels     = {}
all_event_ends = {}

for etf in ETFS:
    lbl, ev = triple_barrier_label(
        etf_close[etf].values, spy_vol_arr,
        tp_mult=TP_MULT, sl_mult=SL_MULT,
        vert_bars=VERT_BARS, dates=dates_arr,
    )
    all_labels[etf]     = pd.Series(lbl, index=close.index)
    all_event_ends[etf] = pd.Series(ev,  index=close.index)

    valid = ~np.isnan(lbl)
    n_v   = valid.sum()
    print(f'  {etf}: n={n_v:,} | '
          f'TP={(lbl[valid] == 1).mean():.1%} | '
          f'SL={(lbl[valid] == -1).mean():.1%} | '
          f'Vert={(lbl[valid] == 0).mean():.1%}')

labels_df     = pd.DataFrame(all_labels)
event_ends_df = pd.DataFrame(all_event_ends)

## 5. Construcción del Dataset y Walk-Forward Splits

In [ ]:
def build_panel(etfs, feature_groups, neutralized, macro, labels_df, event_ends_df):
    """Stack ETF cross-sections into a panel dataset."""
    X_list, y_list, ev_list = [], [], []

    for etf in etfs:
        frames = []
        for name, df in feature_groups.items():
            frames.append(df[etf].rename(name))
        for name, df in neutralized.items():
            frames.append(df[etf].rename(name))
        for col in macro.columns:
            frames.append(macro[col])

        X_etf  = pd.concat(frames, axis=1)
        y_etf  = (labels_df[etf] == 1).astype(float)
        ev_etf = event_ends_df[etf]

        idx   = X_etf.index.intersection(y_etf.index)
        valid = labels_df[etf].loc[idx].notna()

        X_list.append(X_etf.loc[idx][valid])
        y_list.append(y_etf.loc[idx][valid])
        ev_list.append(ev_etf.loc[idx][valid])

    X  = pd.concat(X_list).sort_index()
    y  = pd.concat(y_list).sort_index()
    ev = pd.concat(ev_list).sort_index()
    return X, y, ev


print('Building panel dataset...')
X_panel, y_panel, ev_panel = build_panel(
    ETFS, feature_groups, neutralized, macro, labels_df, event_ends_df
)

oos_mask = X_panel.index >= OOS_START
X_is  = X_panel[~oos_mask].copy()
y_is  = y_panel[~oos_mask].copy()
ev_is = ev_panel[~oos_mask].copy()
X_oos = X_panel[oos_mask].copy()
y_oos = y_panel[oos_mask].copy()

# Impute with IS column medians
col_medians = X_is.median()
X_is.fillna(col_medians, inplace=True)
X_oos.fillna(col_medians, inplace=True)

feature_names = X_is.columns.tolist()

print(f'  IS  : {len(X_is):,} rows | '
      f'{X_is.index[0].date()} → {X_is.index[-1].date()} | '
      f'pos rate={y_is.mean():.1%}')
print(f'  OOS : {len(X_oos):,} rows | '
      f'{X_oos.index[0].date()} → {X_oos.index[-1].date()} [UNTOUCHED]')
print(f'  Features: {len(feature_names)}')

# ── Purged Walk-Forward Expanding Window Splits ────────────────────────────────
def wf_splits(dates, event_ends, val_months, min_train_years, oos_start):
    """
    Purged expanding-window walk-forward splits.
    Purging criterion: training samples where event_end >= val_start are removed.
    """
    min_cut   = dates[0] + pd.DateOffset(years=min_train_years)
    val_start = min_cut
    splits    = []

    while True:
        val_end = val_start + pd.DateOffset(months=val_months)
        if val_end > pd.Timestamp(oos_start):
            break

        tr_mask  = (dates < val_start) & (event_ends < val_start)
        val_mask = (dates >= val_start) & (dates < val_end)

        if tr_mask.sum() > 200 and val_mask.sum() > 20:
            splits.append((
                np.where(tr_mask)[0],
                np.where(val_mask)[0],
                val_start, val_end,
            ))
        val_start = val_end

    return splits


splits = wf_splits(X_is.index, ev_is, VAL_MONTHS, MIN_TRAIN_YEARS, OOS_START)
print(f'\nWalk-forward splits: {len(splits)}')
for i, (tr, val, vs, ve) in enumerate(splits[:3]):
    print(f'  Fold {i+1}: train={len(tr):,} | val={len(val):,} | {vs.date()} → {ve.date()}')
if len(splits) > 3:
    print(f'  ... ({len(splits)-3} more folds)')

## 6. EBM — Optimización de Hiperparámetros (Purged Walk-Forward)

In [ ]:
def ap_lift(y_true, y_prob):
    """Average Precision / positive class frequency."""
    freq = float(y_true.mean())
    if freq == 0 or y_true.sum() == 0:
        return 0.0
    return average_precision_score(y_true, y_prob) / freq


# ── Heuristic base params ─────────────────────────────────────────────────────
EBM_BASE = dict(
    max_bins=256, max_interaction_bins=32, interactions=10,
    outer_bags=8,  inner_bags=0, learning_rate=0.01,
    validation_size=0.15, min_samples_leaf=2, max_leaves=3,
    early_stopping_rounds=50, n_jobs=-1, random_state=RANDOM_STATE,
)

# Lighter params for CI speed
EBM_FAST = dict(
    max_bins=64, max_interaction_bins=16, interactions=3,
    outer_bags=2, inner_bags=0, learning_rate=0.05,
    validation_size=0.15, min_samples_leaf=5, max_leaves=3,
    early_stopping_rounds=20, n_jobs=-1, random_state=RANDOM_STATE,
)

base_params = EBM_FAST if FAST_MODE else EBM_BASE

PARAM_GRID = (
    [
        dict(interactions=3, max_leaves=3, learning_rate=0.05),
        dict(interactions=5, max_leaves=3, learning_rate=0.02),
        dict(interactions=3, max_leaves=5, learning_rate=0.05),
    ] if FAST_MODE else [
        dict(max_bins=128, interactions=5,  max_leaves=3, learning_rate=0.01),
        dict(max_bins=256, interactions=10, max_leaves=3, learning_rate=0.01),
        dict(max_bins=256, interactions=10, max_leaves=5, learning_rate=0.01),
        dict(max_bins=256, interactions=15, max_leaves=3, learning_rate=0.005),
        dict(max_bins=512, interactions=10, max_leaves=3, learning_rate=0.01),
        dict(max_bins=256, interactions=10, max_leaves=3, learning_rate=0.02),
    ]
)

X_arr = X_is.values
y_arr = y_is.values

config_fold_scores = np.full((len(PARAM_GRID), len(splits)), np.nan)

print(f"Running {len(PARAM_GRID)} configs x {len(splits)} folds "
      f"({'FAST' if FAST_MODE else 'FULL'} mode)...\n")

best_cfg_idx = 0
best_mean    = -np.inf

for ci, overrides in enumerate(PARAM_GRID):
    params = {**base_params, **overrides}
    fold_scores = []

    for fi, (tr_idx, val_idx, vs, ve) in enumerate(splits):
        ebm = ExplainableBoostingClassifier(**params)
        ebm.fit(X_arr[tr_idx], y_arr[tr_idx])
        score = ap_lift(y_arr[val_idx], ebm.predict_proba(X_arr[val_idx])[:, 1])
        fold_scores.append(score)
        config_fold_scores[ci, fi] = score

    mean_score = np.nanmean(fold_scores)
    print(f'Config {ci+1}: {overrides}  →  '
          f'AP Lift = {mean_score:.4f} ± {np.nanstd(fold_scores):.4f}')

    if mean_score > best_mean:
        best_mean    = mean_score
        best_cfg_idx = ci

best_overrides = PARAM_GRID[best_cfg_idx]
best_params    = {**base_params, **best_overrides}
print(f'\nBest config [{best_cfg_idx+1}]: {best_overrides}')
print(f'Best AP Lift: {best_mean:.4f}')

## 7. Modelo Final + Probabilidad de Backtest Overfitting (PBO)

In [ ]:
# ── Final model trained on full IS ────────────────────────────────────────────
print('Training final EBM on full IS data...')
final_ebm = ExplainableBoostingClassifier(**best_params)
final_ebm.fit(X_arr, y_arr)

is_ap_lift = ap_lift(y_arr, final_ebm.predict_proba(X_arr)[:, 1])
print(f'IS AP Lift (optimistic bias expected): {is_ap_lift:.4f}')

# ── PBO via CSCV (López de Prado) ─────────────────────────────────────────────
def compute_pbo(cfg_fold_scores, n_sub=None):
    """
    Combinatorially Symmetric Cross-Validation.

    cfg_fold_scores : (n_configs, n_folds) performance matrix
    n_sub           : number of sub-periods to use

    Returns
    -------
    pbo       : float — Probability of Backtest Overfitting
    oos_ranks : ndarray — OOS rank per CSCV combination
    """
    n_cfg, n_folds = cfg_fold_scores.shape
    S = min(n_folds, n_sub or n_folds)
    scores = cfg_fold_scores[:, :S]

    if n_cfg < 2 or S < 4:
        print(f'Warning: need >= 2 configs and >= 4 sub-periods (got {n_cfg}, {S}).')
        return np.nan, np.array([])

    S_half = S // 2
    combos = list(combinations(range(S), S_half))
    oos_ranks = []

    for is_idx in combos:
        oos_idx = [i for i in range(S) if i not in is_idx]
        is_sc   = scores[:, list(is_idx)].mean(axis=1)
        oos_sc  = scores[:, oos_idx].mean(axis=1)

        best     = np.argmax(is_sc)
        oos_rank = float(np.mean(oos_sc <= oos_sc[best]))
        oos_ranks.append(oos_rank)

    oos_ranks = np.array(oos_ranks)
    pbo = float(np.mean(oos_ranks < 0.5))
    return pbo, oos_ranks


pbo, oos_ranks = compute_pbo(config_fold_scores, n_sub=PBO_SUBPERIODS)

print(f"\n{'='*55}")
print(f'Probability of Backtest Overfitting (PBO): {pbo:.4f}')
if not np.isnan(pbo):
    print(f'  Based on {len(oos_ranks)} CSCV combinations')
    risk = 'HIGH RISK (> 50%)' if pbo > 0.5 else 'LOW RISK (<= 50%)'
    print(f'  Assessment: {risk}')
print(f"{'='*55}")

## 8. Cluster Permutation Importance

In [ ]:
def cluster_permutation_importance(
    model, X, y, feat_names, n_clusters=8, n_repeats=5, random_state=RANDOM_STATE
):
    """
    Cluster features by correlation (hierarchical), then permute each cluster
    and measure AP Lift drop.
    """
    rng      = np.random.default_rng(random_state)
    baseline = ap_lift(y, model.predict_proba(X)[:, 1])

    corr = np.corrcoef(X.T)
    np.fill_diagonal(corr, 1.0)
    dist = np.sqrt(np.clip((1 - corr) / 2, 0, 1))
    link = linkage(squareform(dist, checks=False), method='ward')
    clust_ids = fcluster(link, n_clusters, criterion='maxclust')

    results = []
    for cid in range(1, n_clusters + 1):
        feat_idx = np.where(clust_ids == cid)[0]
        if len(feat_idx) == 0:
            continue

        perm_scores = []
        for _ in range(n_repeats):
            X_p = X.copy()
            perm = rng.permutation(len(X_p))
            X_p[:, feat_idx] = X_p[perm][:, feat_idx]
            perm_scores.append(ap_lift(y, model.predict_proba(X_p)[:, 1]))

        top5 = ', '.join(feat_names[i] for i in feat_idx[:5])
        results.append(dict(
            cluster    = cid,
            n_feats    = len(feat_idx),
            importance = baseline - float(np.mean(perm_scores)),
            std        = float(np.std(perm_scores)),
            top_features = top5,
        ))

    imp_df = pd.DataFrame(results).sort_values('importance', ascending=False)
    return imp_df, clust_ids, link, corr


_, val_last, _, _ = splits[-1]
X_val_imp = X_arr[val_last]
y_val_imp = y_arr[val_last]

print('Computing cluster permutation importance (last validation fold)...')
imp_df, clust_ids, linkage_mat, corr_mat = cluster_permutation_importance(
    final_ebm, X_val_imp, y_val_imp,
    feat_names=feature_names, n_clusters=8, n_repeats=5,
)

print('\nCluster Permutation Importance:')
print(imp_df[['cluster', 'n_feats', 'importance', 'std', 'top_features']].to_string(index=False))

## 9. Visualizaciones

In [ ]:
# ── Plot 1: Cluster Permutation Importance + Dendrogram ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
colors_bar = ['#27ae60' if v >= 0 else '#c0392b' for v in imp_df['importance']]
ax.barh(
    range(len(imp_df)), imp_df['importance'],
    xerr=imp_df['std'], color=colors_bar,
    edgecolor='k', alpha=0.85,
    error_kw=dict(ecolor='k', capsize=4),
)
ax.set_yticks(range(len(imp_df)))
ax.set_yticklabels(
    [f"C{r['cluster']} ({r['n_feats']} feats)" for _, r in imp_df.iterrows()],
    fontsize=10,
)
ax.invert_yaxis()
ax.axvline(0, color='k', lw=1)
ax.set_xlabel('AP Lift Drop (higher = more important)', fontsize=11)
ax.set_title('Cluster Permutation Importance', fontsize=13, fontweight='bold')

ax2 = axes[1]
short_names = [f[:10] for f in feature_names]
dendrogram(
    linkage_mat, labels=short_names,
    leaf_rotation=90, leaf_font_size=5, ax=ax2,
    color_threshold=0.7 * linkage_mat[:, 2].max(),
)
ax2.set_title('Feature Hierarchy Dendrogram', fontsize=13, fontweight='bold')
ax2.set_xlabel('Features', fontsize=10)
ax2.set_ylabel('Distance', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cluster_permutation_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: cluster_permutation_importance.png')

# ── Plot 2: PBO distribution ──────────────────────────────────────────────────
if len(oos_ranks) > 0:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(oos_ranks, bins=20, edgecolor='k', alpha=0.7, color='steelblue')
    ax.axvline(0.5, color='red', ls='--', lw=2, label='rank = 0.5')
    ax.set_xlabel('OOS Rank of IS-Optimal Strategy', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title(f'CSCV Distribution  –  PBO = {pbo:.2%}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.text(0.05, 0.95, f'PBO = {pbo:.2%}', transform=ax.transAxes,
            fontsize=14, va='top',
            bbox=dict(boxstyle='round', fc='wheat', alpha=0.6))
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'pbo_distribution.png', dpi=150, bbox_inches='tight')
    plt.close()
    print('Saved: pbo_distribution.png')

# ── Plot 3: Walk-Forward results ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(range(1, len(splits) + 1), config_fold_scores[best_cfg_idx],
        'o-', color='steelblue', lw=2, ms=7)
ax.axhline(1.0, color='grey', ls='--', alpha=0.5, label='AP Lift = 1 (random)')
ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('AP Lift', fontsize=12)
ax.set_title('Walk-Forward AP Lift – Best Config', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
means = np.nanmean(config_fold_scores, axis=1)
stds  = np.nanstd(config_fold_scores, axis=1)
colors_cfg = ['#27ae60' if i == best_cfg_idx else 'steelblue'
              for i in range(len(PARAM_GRID))]
ax.barh(
    range(len(PARAM_GRID)), means, xerr=stds,
    color=colors_cfg, edgecolor='k', alpha=0.85,
    error_kw=dict(ecolor='k', capsize=4),
)
ax.set_yticks(range(len(PARAM_GRID)))
ax.set_yticklabels([f'C{i+1}: {list(p.values())}' for i, p in enumerate(PARAM_GRID)], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Mean AP Lift', fontsize=12)
ax.set_title('Config Comparison (green = best)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'walkforward_results.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: walkforward_results.png')

# ── Plot 4: Label distribution ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
w = 0.25
x = np.arange(len(ETFS))
tp_r   = (labels_df == 1).sum()  / labels_df.notna().sum()
sl_r   = (labels_df == -1).sum() / labels_df.notna().sum()
vert_r = (labels_df == 0).sum()  / labels_df.notna().sum()

ax.bar(x - w, tp_r[ETFS],   w, label='TP (+1)',      color='#27ae60', edgecolor='k', alpha=0.85)
ax.bar(x,     vert_r[ETFS], w, label='Vertical (0)', color='#95a5a6', edgecolor='k', alpha=0.85)
ax.bar(x + w, sl_r[ETFS],   w, label='SL (-1)',      color='#c0392b', edgecolor='k', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ETFS, rotation=45)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Triple Barrier Hit Rate by Sector ETF', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'label_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: label_distribution.png')

## 10. Resumen del Pipeline

In [ ]:
sep = '=' * 60
print(sep)
print('PIPELINE SUMMARY — EBM Sector ETF Triple Barrier')
print(sep)
print(f'\nData:')
print(f'  ETFs     : {", ".join(ETFS)}')
print(f'  IS period: {START_DATE} → {OOS_START}')
print(f'  OOS      : {OOS_START} → {END_DATE}  [UNTOUCHED]')
print(f'  IS rows  : {len(X_is):,}')
print(f'  Features : {len(feature_names)}')
print(f'\nTriple Barrier:')
print(f'  TP = {TP_MULT}x  |  SL = {SL_MULT}x  |  Vol window = {VOL_WINDOW}d  |  Vertical = {VERT_BARS} sessions')
print(f'  IS positive rate: {y_is.mean():.1%}')
print(f'\nWalk-Forward CV:')
print(f'  Splits    : {len(splits)}')
print(f'  Val window: {VAL_MONTHS} months (expanding train, purging event_end >= val_start)')
print(f'  Best cfg  : {best_overrides}')
print(f'  Best mean AP Lift: {best_mean:.4f}')
print(f'\nBacktest Overfitting (CSCV):')
print(f'  PBO = {pbo:.2%}  ({"HIGH RISK" if pbo > 0.5 else "LOW RISK"})')
print(f'  CSCV combinations: {len(oos_ranks)}')
print(f'\nOutputs → {OUTPUT_DIR}/')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name}')
print(sep)